# YOLO — You Only Look Once: Unified, Real-Time Object Detection (2015)

_Papers · Computer Vision_

**Detect every object in one network pass by predicting a grid of boxes plus classes at once.**

---

This notebook is a practice scaffold for the **YOLO — You Only Look Once: Unified, Real-Time Object Detection (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
S, B, C = 7, 2, 20          # Pascal VOC grid: 7x7 cells, 2 boxes/cell, 20 classes

# --- 0. A tiny conv stem that outputs the S x S x (B*5 + C) grid tensor (imported plumbing). ---
class TinyYOLOHead(nn.Module):
    def __init__(self, S, B, C):
        super().__init__()
        self.S, self.depth = S, B * 5 + C
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),   # 448 -> 224
            nn.AdaptiveAvgPool2d(S),                               # -> S x S
            nn.Conv2d(16, self.depth, 1))                          # -> depth channels
    def forward(self, x):
        return self.net(x).permute(0, 2, 3, 1)   # (1, S, S, B*5+C)

head = TinyYOLOHead(S, B, C)
img  = torch.rand(1, 3, 448, 448)               # one 448x448 image (paper's input size)
grid = head(img)
print("grid tensor shape:", tuple(grid.shape), " (expect (1, 7, 7, 30))")

# --- 1. Decode one cell's box: offsets (x,y in 0..1) + image-relative (w,h) -> absolute corners. ---
def decode_cell(x, y, w, h, row, col, S):
    xc = (col + x) / S                          # add column offset, normalize by grid
    yc = (row + y) / S                          # add row offset
    return [xc - w/2, yc - h/2, xc + w/2, yc + h/2]   # [x1, y1, x2, y2]

P = decode_cell(0.6, 0.4, 0.30, 0.20, row=3, col=2, S=S)
print("decoded P [x1,y1,x2,y2] =", [round(v, 3) for v in P])
# decoded P = [0.221, 0.386, 0.521, 0.586]

# --- 2. IoU = overlap / union (clamp overlap at 0). Recap of dl-object-detection. ---
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    areaA = (a[2]-a[0]) * (a[3]-a[1]); areaB = (b[2]-b[0]) * (b[3]-b[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

Q = [0.271, 0.386, 0.571, 0.586]                # neighbour box shifted right by 0.05
print("IoU(P, Q) =", round(iou(P, Q), 3), " (expect ~0.714, prints 0.715 at 3dp)")

# --- 3. Class-specific confidence (Eqn. 1): box confidence * cell class prob. ---
def class_conf(box_conf, class_probs):
    return [box_conf * p for p in class_probs]
print("class-specific scores =", [round(s, 3) for s in class_conf(0.9, [0.8, 0.1, 0.1])])
# = [0.72, 0.09, 0.09]  -> the box's score for each class

# --- 4. Non-max suppression: keep top score, drop boxes overlapping it above threshold. ---
def nms(boxes, scores, thr=0.5):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) <= thr]
    return keep

R = [0.80, 0.80, 0.95, 0.95]                    # a far-off, non-overlapping box
boxes  = [P, Q, R]; scores = [0.90, 0.75, 0.80]
print("NMS keeps indices:", nms(boxes, scores, thr=0.5), " (expect [0, 2] -> P and R; Q dropped)")

# --- 5. ABLATION: detection WITH vs WITHOUT NMS -> count boxes per object. ---
# Two objects, each covered by 2 overlapping boxes (duplicates).
obj1a = [0.20,0.20,0.50,0.60]; obj1b = [0.22,0.22,0.52,0.62]   # cluster 1 (IoU high)
obj2a = [0.60,0.20,0.90,0.60]; obj2b = [0.62,0.22,0.92,0.62]   # cluster 2 (IoU high)
ab_boxes  = [obj1a, obj1b, obj2a, obj2b]
ab_scores = [0.92, 0.81, 0.88, 0.70]
print("\nWITHOUT NMS: boxes kept =", len(ab_boxes), " (4 -> objects double-counted)")
print("WITH    NMS: boxes kept =", len(nms(ab_boxes, ab_scores, thr=0.5)),
      " (2 -> one clean box per object)")
# Our toy run, not the paper's number. The paper reports NMS "adds 2-3% in mAP" (Sec 2.3).

## Visualize the data & results

_How many boxes survive per object WITH non-max suppression vs WITHOUT, as more duplicate boxes pile up?_

In [ ]:
# Reproduce the qualitative effect: NMS collapses each duplicate cluster to one box.
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    aA = (a[2]-a[0])*(a[3]-a[1]); aB = (b[2]-b[0])*(b[3]-b[1])
    u = aA + aB - inter
    return inter / u if u > 0 else 0.0

def nms(boxes, scores, thr=0.5):
    order = sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) <= thr]
    return keep

centers = [(0.35, 0.40), (0.75, 0.40)]    # two true objects
without, with_nms = [], []
for dup in range(1, 5):                   # 1x .. 4x duplicate boxes per object
    boxes, scores = [], []
    for k, (cx, cy) in enumerate(centers):
        for d in range(dup):
            off = 0.01 * d                # tiny jitter -> high mutual IoU
            boxes.append([cx-0.15+off, cy-0.20+off, cx+0.15+off, cy+0.20+off])
            scores.append(0.9 - 0.05*d - 0.01*k)
    without.append(len(boxes))
    with_nms.append(len(nms(boxes, scores, thr=0.5)))
print("duplicates per object :", [1, 2, 3, 4])
print("boxes WITHOUT NMS     :", without)   # [2, 4, 6, 8]
print("boxes WITH    NMS     :", with_nms)  # [2, 2, 2, 2]
# Our toy run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Decode a cell. On a $S=7$ grid, cell (row $1$, col $5$) predicts centre offset
            $x=0.2$, $y=0.8$ and size $w=0.14$, $h=0.28$. Give the box's absolute centre and its corner
            coordinates $[x_1,y_1,x_2,y_2]$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Centre: $x_{\text{img}}=(5+0.2)/7=5.2/7\approx0.743$, $y_{\text{img}}=(1+0.8)/7=1.8/7\approx0.257$. — _$x,y$ are offsets inside the cell; add the column/row and divide by $S$ to get image coordinates._
- Half-sizes: $w/2=0.07$, $h/2=0.14$ (already image-relative). — _Width/height are fractions of the whole image, so they are not divided by $S$._
- Corners: $x_1=0.743-0.07=0.673$, $x_2=0.743+0.07=0.813$; $y_1=0.257-0.14=0.117$, $y_2=0.257+0.14=0.397$. — _A box centred at $(x,y)$ with width $w$, height $h$ spans $[x-w/2,x+w/2]\times[y-h/2,y+h/2]$._

**Answer:** Centre $\approx(0.743,\,0.257)$; corners
                 $[x_1,y_1,x_2,y_2]\approx[0.673,\,0.117,\,0.813,\,0.397]$. The cell offset moves the box
                 to the right side of the image; $w,h$ stay image-relative.

</details>

**Problem 2.** IoU. Box A $=[0.20,0.20,0.50,0.60]$ and box B $=[0.30,0.30,0.60,0.70]$ (all
            $[x_1,y_1,x_2,y_2]$). Compute their IoU.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Areas: $A=(0.50-0.20)(0.60-0.20)=0.30\times0.40=0.12$; $B=(0.60-0.30)(0.70-0.30)=0.30\times0.40=0.12$. — _Box area is width times height from the corners._
- Overlap width $=\min(0.50,0.60)-\max(0.20,0.30)=0.50-0.30=0.20$; overlap height $=\min(0.60,0.70)-\max(0.20,0.30)=0.60-0.30=0.30$. — _The intersection rectangle runs from the larger left/top edge to the smaller right/bottom edge; clamp at 0 if negative._
- Intersection $=0.20\times0.30=0.06$; union $=0.12+0.12-0.06=0.18$; IoU $=0.06/0.18\approx0.333$. — _Union subtracts the shared area so it is not counted twice._

**Answer:** $\text{IoU}=0.06/0.18=1/3\approx0.333$. They overlap moderately &mdash; below a typical
                 NMS threshold of $0.5$, so NMS would keep both as separate detections.

</details>

**Problem 3.** The NMS ablation. You run the toy detector and get four boxes for what is really two
            objects: two boxes tightly overlapping object 1 (scores $0.92$ and $0.81$, IoU $0.78$) and two
            tightly overlapping object 2 (scores $0.88$ and $0.70$, IoU $0.65$); cross-object IoUs are $0$.
            With NMS threshold $0.5$: how many boxes survive WITH NMS vs WITHOUT, and what does that show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- WITHOUT NMS: keep all four boxes. — _No suppression step runs, so every above-threshold detection is reported &mdash; including the duplicates._
- WITH NMS: keep $0.92$ (top), drop $0.81$ (IoU $0.78\gt0.5$ with it); then keep $0.88$, drop $0.70$ (IoU $0.65\gt0.5$). — _NMS keeps the highest-scoring box per cluster and removes overlapping lower-scoring boxes._
- Result: 2 boxes with NMS, 4 without &mdash; one clean box per object vs doubled detections. — _The paper notes NMS "adds 2-3% in mAP" by removing exactly these duplicates (&sect;2.3)._

**Answer:** With NMS: 2 boxes; without: 4. NMS collapses each over-counted cluster to its
                 single best box, so each object is reported once. Turning NMS off doubles the detections
                 here &mdash; the ablation reproduces, on toy data, why the paper keeps NMS (it "adds 2-3%
                 in mAP"). The CODEVIZ panel shows this count drop.

</details>